# SCDB Vote Prediction
Predict a justice's **vote** given the justice identity and case type.

**Target:** `vote` — how a justice voted (1 = majority, 2 = dissent, etc.)  
**Key features:** `justice`, `issueArea` (case type), plus other case-level attributes

**Models compared:**
1. Decision Tree
2. Random Forest
3. K-Nearest Neighbors (KNN)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score
)

## 1. Load Data

In [ ]:
DATA_PATH = r"C:\Users\isabe\OneDrive\Desktop\DS-Studio-II\datasets\SCDB_2025_01_justiceCentered_LegalProvision.csv.zip"

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

## 2. Select Features & Target

We keep only columns relevant to predicting a justice's vote:
- **`justice`** / **`justiceName`** — who is voting
- **`issueArea`** — broad category of the case (Criminal Procedure, Civil Rights, etc.)
- Case-level context: `decisionDirection`, `lcDisposition`, `certReason`, `jurisdiction`, `partyWinning`, `majVotes`, `minVotes`

Columns dropped: IDs, free-text citations, dates, and any post-decision fields that would leak the outcome.

In [ ]:
KEEP_COLS = [
    'justice',
    'justiceName',
    'issueArea',
    'decisionDirection',
    'lcDisposition',
    'lcDispositionDirection',
    'certReason',
    'jurisdiction',
    'partyWinning',
    'majVotes',
    'minVotes',
    'decisionType',
    'vote',
]

df = df[KEEP_COLS].copy()
print(f"Working shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")

## 3. Clean & Encode

In [ ]:
# Drop rows with missing target
df = df.dropna(subset=['vote'])

# Encode justiceName (string -> integer)
df['justiceName'] = pd.factorize(df['justiceName'])[0]

# Fill remaining NaNs with median
df = df.fillna(df.median(numeric_only=True))

print(f"Clean shape: {df.shape}")
print(f"\nVote distribution:\n{df['vote'].value_counts()}")

## 4. Train / Test Split

In [ ]:
X = df.drop(columns=['vote'])
y = df['vote']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaled version for KNN (distance-based, sensitive to feature scale)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

## 5. Train All Three Models

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=20, random_state=42, n_jobs=-1),
    'KNN':           KNeighborsClassifier(n_neighbors=9, n_jobs=-1),
}

# KNN uses scaled data; tree-based models use raw data
train_data = {
    'Decision Tree': (X_train,        X_test),
    'Random Forest': (X_train,        X_test),
    'KNN':           (X_train_scaled, X_test_scaled),
}

predictions = {}
for name, model in models.items():
    Xtr, Xte = train_data[name]
    model.fit(Xtr, y_train)
    predictions[name] = model.predict(Xte)
    print(f"[{name}] trained.")

## 6. Evaluation — Accuracy, Precision, Recall & F1

In [ ]:
rows = []
for name, y_pred in predictions.items():
    rows.append({
        'Model':                name,
        'Accuracy':             accuracy_score(y_test, y_pred),
        'Precision (weighted)': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'Recall (weighted)':    recall_score(y_test, y_pred,    average='weighted', zero_division=0),
        'F1 (weighted)':        f1_score(y_test, y_pred,        average='weighted', zero_division=0),
        'Precision (macro)':    precision_score(y_test, y_pred, average='macro',    zero_division=0),
        'Recall (macro)':       recall_score(y_test, y_pred,    average='macro',    zero_division=0),
        'F1 (macro)':           f1_score(y_test, y_pred,        average='macro',    zero_division=0),
    })

summary = pd.DataFrame(rows).set_index('Model').round(4)
print(summary.to_string())

## 7. Classification Reports (per class)

In [ ]:
for name, y_pred in predictions.items():
    print(f"{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, zero_division=0))

## 8. Confusion Matrices

In [ ]:
labels = sorted(y.unique())
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(name)

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. Cross-Validation (5-fold)

In [ ]:
CV_SCORING = ['accuracy', 'f1_weighted', 'precision_weighted', 'recall_weighted']
METRIC_LABELS = ['Accuracy', 'F1 (weighted)', 'Precision (weighted)', 'Recall (weighted)']
SCORE_KEYS    = ['accuracy', 'f1_weighted', 'precision_weighted', 'recall_weighted']

# KNN uses a Pipeline so scaling is applied within each fold (avoids data leakage)
cv_models = {
    'Decision Tree': models['Decision Tree'],
    'Random Forest': models['Random Forest'],
    'KNN':           Pipeline([('scaler', StandardScaler()),
                               ('knn', KNeighborsClassifier(n_neighbors=9, n_jobs=-1))]),
}

cv_all = {}
for name, model in cv_models.items():
    cv_all[name] = cross_validate(
        model, X, y,
        cv=5,
        scoring=CV_SCORING,
        return_train_score=True,
        n_jobs=-1
    )
    print(f"[{name}] CV done.")

In [ ]:
# Summary table: test mean +/- std for each model and metric
cv_rows = []
for name, results in cv_all.items():
    row = {'Model': name}
    for label, key in zip(METRIC_LABELS, SCORE_KEYS):
        mean = results[f'test_{key}'].mean()
        std  = results[f'test_{key}'].std()
        row[label] = f"{mean:.4f} +/- {std:.4f}"
    cv_rows.append(row)

cv_summary = pd.DataFrame(cv_rows).set_index('Model')
print(cv_summary.to_string())

In [ ]:
# Per-fold accuracy bar chart for all three models
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

for ax, (name, results) in zip(axes, cv_all.items()):
    fold_accs = results['test_accuracy']
    ax.bar(range(1, 6), fold_accs, color='steelblue', edgecolor='black')
    ax.axhline(fold_accs.mean(), color='red', linestyle='--',
               label=f'Mean = {fold_accs.mean():.4f}')
    ax.set_title(name)
    ax.set_xlabel('Fold')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

plt.suptitle('5-Fold CV — Test Accuracy per Fold', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Model Comparison Chart

In [ ]:
comp = summary[['Accuracy', 'F1 (weighted)', 'Precision (weighted)', 'Recall (weighted)']]

comp.T.plot(kind='bar', figsize=(10, 5), edgecolor='black')
plt.title('Model Comparison — Test Set Metrics')
plt.ylabel('Score')
plt.xticks(rotation=20)
plt.ylim(0, 1)
plt.legend(title='Model', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## 11. Feature Importance (Decision Tree & Random Forest)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, name in zip(axes, ['Decision Tree', 'Random Forest']):
    fi = pd.Series(models[name].feature_importances_, index=X.columns).sort_values(ascending=False)
    sns.barplot(x=fi.values, y=fi.index, ax=ax)
    ax.set_title(f'Feature Importances — {name}')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 12. Visualize Decision Tree (top 3 levels)

In [ ]:
plt.figure(figsize=(18, 8))
plot_tree(models['Decision Tree'], max_depth=3, feature_names=X.columns.tolist(), filled=True, fontsize=8)
plt.title('Decision Tree (depth <= 3)')
plt.tight_layout()
plt.show()

## 13. EDA — Vote Counts by Issue Area

In [ ]:
pivot = df.groupby(['issueArea', 'vote']).size().unstack(fill_value=0)
pivot.plot(kind='bar', stacked=True, figsize=(12, 5), colormap='tab10')
plt.title('Vote Counts by Issue Area')
plt.xlabel('Issue Area')
plt.ylabel('Count')
plt.legend(title='Vote', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()